# 🐜 ANT — A100 Training

**828K parameter byte-level transformer** with persistent external memory and memory-based chat.

## 3-Phase Curriculum
1. **Phase 1** — Language Model (Wikipedia + shell, sliding window)
2. **Phase 2** — LM + Memory QA (bAbI with cross-attention, 100% target)
3. **Phase 3** — LM + QA + Memory-Based Chat (user→memory, agent→sliding generation)

## A100 Optimizations
- **BF16 mixed precision** (Tensor Cores)
- **torch.compile** (inductor kernel fusion)
- **Large batches** (B=128 with grad_accum=2 = 256 effective)
- **Checkpoints auto-upload to HuggingFace Hub**

## 1. Setup

In [ ]:
# Verify GPU and install dependencies
!nvidia-smi
!pip install -q datasets numpy torch huggingface_hub

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    print(f'BF16: {torch.cuda.is_bf16_supported()}')

In [ ]:
# Clone repo and login to HuggingFace
import os

REPO_DIR = '/content/ANT'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kaaninel/ANT.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
os.chdir(REPO_DIR)

# HuggingFace login (set HF_TOKEN in Colab secrets)
from huggingface_hub import login, create_repo
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()

HF_REPO = 'kaaninel/ANT'
try:
    create_repo(HF_REPO, exist_ok=True, repo_type='model')
    print(f'\u2713 HF repo: https://huggingface.co/{HF_REPO}')
except: pass

## 2. Configuration

Edit these values and re-run to adjust. The A100 handles much larger batches than M4.

In [ ]:
# ================================================================
# TRAINING CONFIG
# ================================================================

# Steps per phase
PHASE1_STEPS = 2000   # Phase 1: LM only (sliding window)
PHASE2_STEPS = 2000   # Phase 2: LM + Memory QA
PHASE3_STEPS = 6000   # Phase 3: LM + QA + Memory-Based Chat

# Batch size
BATCH_SIZE = 128      # Micro-batch
GRAD_ACCUM = 2        # 128 x 2 = 256 effective batch
STRIDE = 1            # stride=1 for correct LM (every position is a target)

# Data sizes
N_WIKI = 50_000       # Wikipedia sentences
N_SHELL = 5_000       # Shell commands
N_QA = 20_000         # QA training examples
N_CHAT = 30_000       # HF chat pairs (UltraChat + SmolTalk)

# Architecture
WINDOW_SIZE = 8
NUM_PASSES = 4        # 4 passes = 17-byte receptive field per token
EVAL_INTERVAL = 500
LR = 3e-4             # Higher LR for A100 (larger batch stabilizes)

# A100 optimizations
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

total = PHASE1_STEPS + PHASE2_STEPS + PHASE3_STEPS
eff_batch = BATCH_SIZE * GRAD_ACCUM
print(f'Total steps: {total:,}')
print(f'Batch: {BATCH_SIZE} x {GRAD_ACCUM} accum = {eff_batch} effective')
print(f'Window: W={WINDOW_SIZE} passes={NUM_PASSES} stride={STRIDE}')
print(f'Data: wiki={N_WIKI:,} shell={N_SHELL:,} qa={N_QA:,} chat={N_CHAT:,}')
# A100 with BF16 + compile: ~5-10 it/s expected
print(f'Est. time: ~{total / 7 / 60:.0f} min (at ~7 it/s on A100)')

## 3. Load Checkpoint (Optional)

If you have a previous checkpoint on HuggingFace, load it to continue training.

In [ ]:
# Try to download existing checkpoint from HF
import os
CKPT_DIR = '/content/ant_checkpoints/chat'
os.makedirs(CKPT_DIR, exist_ok=True)

RESUME_FROM = None
try:
    from huggingface_hub import hf_hub_download
    for fname in ['checkpoint_best.pt', 'checkpoint_latest.pt']:
        try:
            path = hf_hub_download(
                repo_id='kaaninel/ANT',
                filename=f'checkpoints/chat/{fname}',
                local_dir='/content/ant_hf_cache'
            )
            import shutil
            dest = os.path.join(CKPT_DIR, fname)
            shutil.copy2(path, dest)
            RESUME_FROM = dest
            print(f'Downloaded {fname} from HF')
            break
        except Exception as e:
            print(f'{fname}: {e}')
except ImportError:
    pass

if RESUME_FROM:
    print(f'Will resume from: {RESUME_FROM}')
else:
    print('Starting from scratch (no checkpoint found)')

## 4. Train

All 3 phases run automatically. Checkpoints upload to HuggingFace every 2000 steps.

In [ ]:
import torch

from config import ModelConfig
from model import ANT
from train_micro import (
    VOCAB_SIZE, train_chat, log_model_summary,
)

device = 'cuda'

# Build model
cfg = ModelConfig()
cfg.vocab_size = VOCAB_SIZE
model = ANT(cfg, use_checkpoint=False).to(device)
log_model_summary(model, cfg)

# Load checkpoint if available
if RESUME_FROM and os.path.exists(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, map_location=device, weights_only=False)
    if 'model' in ckpt:
        state = ckpt['model']
        # Strip torch.compile prefix if present
        if any(k.startswith('_orig_mod.') for k in state):
            state = {k.replace('_orig_mod.', '', 1): v for k, v in state.items()}
        model.load_state_dict(state)
        print(f'Loaded step={ckpt.get("step", 0)}, QA={ckpt.get("accuracy", 0):.1%}')

# Train all phases
best_acc = train_chat(
    model, cfg, device,
    output_dir=CKPT_DIR,
    steps_phase1=PHASE1_STEPS,
    steps_phase2=PHASE2_STEPS,
    steps_phase3=PHASE3_STEPS,
    lr=LR,
    batch_size=BATCH_SIZE,
    grad_accum=GRAD_ACCUM,
    eval_interval=EVAL_INTERVAL,
    window_size=WINDOW_SIZE,
    num_passes=NUM_PASSES,
    stride=STRIDE,
    n_wiki=N_WIKI,
    n_shell=N_SHELL,
    n_qa=N_QA,
    n_chat=N_CHAT,
    use_bf16=True,
    use_compile=True,
    use_hf_chat=True,
    hf_repo='kaaninel/ANT',
    hf_upload_interval=2000,
)

print(f'\nTraining complete! Best QA: {best_acc:.1%}')

## 5. Test Generation

Quick test of memory-based chat and text generation.

In [ ]:
import torch
from config import ModelConfig
from model import ANT
from train_micro import (
    VOCAB_SIZE, BOS_ID, tokenize, detokenize,
    sliding_generate, encode_sentence_frozen,
)

# Load best checkpoint
import os
ckpt_dir = '/content/ant_checkpoints/chat'
best = os.path.join(ckpt_dir, 'checkpoint_best.pt')
if not os.path.exists(best):
    best = os.path.join(ckpt_dir, 'checkpoint_latest.pt')

ckpt = torch.load(best, map_location='cuda', weights_only=False)
cfg = ModelConfig()
cfg.vocab_size = VOCAB_SIZE
model = ANT(cfg, use_checkpoint=False).cuda()

state = ckpt['model']
if any(k.startswith('_orig_mod.') for k in state):
    state = {k.replace('_orig_mod.', '', 1): v for k, v in state.items()}
model.load_state_dict(state)
model.eval()

W = ckpt.get('window_size', 8)
P = ckpt.get('num_passes', 4)
print(f'Loaded step={ckpt["step"]}, QA={ckpt.get("accuracy", 0):.1%}, W={W}, passes={P}')

# --- Text generation (no memory) ---
print('\n=== Text Generation ===')
prompts = ['The ', 'In the year ', '$ ls -la ']
for p in prompts:
    ids = [BOS_ID] + tokenize(p)
    out = sliding_generate(model, ids, max_new_tokens=100,
                           temperature=0.7, top_k=30,
                           window_size=W, num_passes=P)
    print(f'\n--- {repr(p)} ---')
    print(detokenize(out)[:200])

# --- Memory-based chat ---
print('\n=== Memory-Based Chat ===')
questions = ['Hello, how are you?', 'What is the meaning of life?']
for q in questions:
    # Encode question into memory
    q_ids = torch.tensor([tokenize(q)[:190]], dtype=torch.long, device='cuda')
    with torch.no_grad():
        mk, mv, mm = encode_sentence_frozen(model, q_ids, 'cuda')
    
    # Generate with agent tag prompt + memory
    agent_tag = 'localhost/ant/chat@2026-04-08T12:00:00Z: '
    prompt_ids = [BOS_ID] + tokenize(agent_tag)
    out = sliding_generate(model, prompt_ids, max_new_tokens=200,
                           temperature=0.7, top_k=30,
                           window_size=W, num_passes=P,
                           memory_keys=mk, memory_values=mv,
                           memory_mask=mm)
    response = detokenize(out)[len(agent_tag) + 1:]  # strip tag from output
    print(f'\nUser: {q}')
    print(f'ANT:  {response[:300]}')

## 6. Upload Final Checkpoint

In [ ]:
from huggingface_hub import HfApi
import os
api = HfApi()

ckpt_dir = '/content/ant_checkpoints/chat'
for f in ['checkpoint_latest.pt', 'checkpoint_best.pt']:
    path = os.path.join(ckpt_dir, f)
    if os.path.exists(path):
        api.upload_file(
            path_or_fileobj=path,
            path_in_repo=f'checkpoints/chat/{f}',
            repo_id='kaaninel/ANT', repo_type='model')
        print(f'Uploaded {f}')

print('\nDone! Check https://huggingface.co/kaaninel/ANT')